In [1]:
import os
import h5py
import numpy as np
from tqdm import tqdm

import torch

In [2]:
file = "music_raw/samp4000_dur20.hdf5"
data = h5py.File(file, 'r')
print(data.keys())

<KeysViewHDF5 ['audio', 'labels']>


In [3]:
np.random.seed(0)
perm = np.random.permutation(range(len(data['audio'])))

inverse_perm = np.zeros_like(perm)
for i in range(len(perm)):
    inverse_perm[perm[i]] = i

In [4]:
SONGS = data['audio'][:][inverse_perm]
SONGS_id = data['labels'][:][inverse_perm]
SONGS_id

array([   0,    0,    0, ..., 1188, 1188, 1188])

In [21]:
num_songs = max(SONGS_id) + 1

SONGS_OUT = []
for i in tqdm(range(num_songs)):
    locs = np.where(SONGS_id == i)[0]
    SONGS_OUT.append(np.concatenate(SONGS[locs], axis=0).reshape(-1, 400))

100%|██████████| 1189/1189 [00:02<00:00, 589.15it/s]


In [ ]:
SONGS = []
SONGS_id = []
single_dur = 500
samp_rate_override = SONGS_OUT[0].shape[1]
print(f"Splitting songs into {single_dur} length pieces")

for n,song in tqdm(enumerate(SONGS_OUT)):
    pieces = [i for i in np.array_split(song, [single_dur*i for i in range(len(song) // single_dur + 1)]) if i.shape[0] == single_dur]
    SONGS += pieces
    SONGS_id += [n]*len(pieces)

# randomly permuting the corpuses
np.random.seed(0)
perm = np.random.permutation(range(len(SONGS)))
SONGS = np.array(SONGS)[perm]
SONGS_id = np.array(SONGS_id)[perm]

save_dir = "music_processed"
os.makedirs(save_dir, exist_ok=True)

file = h5py.File(os.path.join(save_dir, f"samp{samp_rate_override}_dur{single_dur}.hdf5"), "w")
file.create_dataset("audio", data=SONGS, dtype=np.float64)
file.create_dataset("labels", data=SONGS_id, dtype=int)
file.close()

Splitting songs into 500 length pieces


1189it [00:00, 97399.08it/s]


In [ ]:
data = h5py.File("music_processed/samp400_dur500.hdf5", 'r')
print(data['audio'].shape, data['labels'].shape)
data.close()

(4779, 500, 400) (4779,)


: 